# Propagational MLP (PMLP) on Cora

Classify papers in the Cora citation network by topic. A PMLP
([Yang et al., 2023](https://arxiv.org/abs/2212.09034)) is trained as a plain MLP that ignores the
graph, and only uses message passing when making predictions. Surprisingly, this works about as well
as a GCN.

Same model as PyG's [`examples/pmlp.py`](https://github.com/pyg-team/pytorch_geometric/blob/master/examples/pmlp.py).

In [ ]:
# Install K3-Node, then choose a backend: "tensorflow", "torch" or "jax"
!pip install -q git+https://github.com/anas-rz/k3-node
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

## Load the data

`data` holds one graph: node features `x`, edges `edge_index`, labels `y`, and masks marking the training, validation and test nodes.

In [ ]:
import keras
from keras import ops
from k3_node.datasets import Planetoid
from k3_node.loader import FullGraphDataset
from k3_node.transforms import NormalizeFeatures

dataset = Planetoid("data/Planetoid", name="Cora", transform=NormalizeFeatures())
data = dataset[0]
print(data)

## Define the model

In [ ]:
from k3_node.models import PMLP


class PMLPClassifier(keras.Model):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.pmlp = PMLP(in_channels, hidden_channels=16, out_channels=out_channels,
                         num_layers=2, dropout=0.5, norm=False)

    def call(self, data, training=False):
        # Train without the graph (like an MLP); use message passing for predictions
        edge_index = None if training else data.edge_index
        return self.pmlp(data.x, edge_index, training=training)


model = PMLPClassifier(dataset.num_features, dataset.num_classes)

## Train

`FullGraphDataset` feeds the whole graph to Keras; `mask` selects which nodes count in the loss and the accuracy.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01, weight_decay=0.0005),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    weighted_metrics=["accuracy"],
)
model.fit(
    FullGraphDataset(data, mask="train_mask"),
    validation_data=FullGraphDataset(data, mask="val_mask"),
    epochs=200,
    verbose=2,
)

## Evaluate

In [ ]:
loss, accuracy = model.evaluate(FullGraphDataset(data, mask="test_mask"), verbose=0)
print(f"Test accuracy: {accuracy:.4f}")